##  Bibliotecas:

In [1]:
import geopandas as gpd
import ee
import pandas as pd
import json
import ast

##  Autenticação API:

In [2]:
ee.Authenticate()

# Ligar API GEE
ee.Initialize()


Successfully saved authorization token.


## Shape de Pontos:

In [3]:
#Ler shape de Contornos 
shapefile_path = (r"C:\Users\sensix\Desktop\SENSIX\BI_dev_team\ssx-rd\Variaveis_Planetarias\Franciosi\Pivos.shp")
poligonos = gpd.read_file(shapefile_path)

## SENTINEL-2:

In [4]:
def maskClouds(image):
    # A banda QA60 contém a máscara de nuvens
    cloudBitMask = ee.Number(2).pow(10).int()
    cirrusBitMask = ee.Number(2).pow(11).int()

    # Seleciona a banda QA60
    qa = image.select('QA60')

    # Aplica a máscara de nuvens
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))

    return image.updateMask(mask)

# Função para calcular a média das bandas
def calculateMean(image):
    mean = image.reduceRegions(
        collection=poligonos_ee,
        reducer=ee.Reducer.mean(),
        scale=10
    )
    return mean

# DataFrame final para armazenar os resultados
df_final = pd.DataFrame()

# Loop que abre uma feição por vez
for index, row in poligonos.iterrows():
    polygon = row['geometry']
    
    # Converter para o tipo lido pelo GEE
    poligonos_ee = ee.Geometry.Polygon(list(polygon.exterior.coords))
    print(row['ID'])
    
    # Definir período de tempo da análise 
    startDate = '2022-01-01'
    endDate = '2023-12-31'

    # Carregar imagens do satélite Sentinel-2 e aplicar a máscara de nuvens
    s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(poligonos_ee).filterDate(startDate, endDate).map(maskClouds)
    
    # Calcular a média das bandas
    mean = s2.map(calculateMean).flatten()

    # Converter o resultado em um DataFrame
    resultados = mean.getInfo()
    data = []
    for feature in resultados['features']:
        properties = feature['properties']
        properties['cena'] = feature['id']
        properties['ID'] = row['ID']
        data.append(properties)

    dataframe = pd.DataFrame(data)
    df_final = pd.concat([df_final, dataframe])

1
2
3
4


## SENTINEL-1

In [12]:
#Armazena as informações das Bandas por Poligono
df_final = pd.DataFrame()

# loop que abre uma feição por vez
for index, row in poligonos.iterrows():
    polygon = row['geometry'] 
    
    # Converter para o tipo lido pelo GEE
    poligonos_ee = ee.Geometry.Polygon(list(polygon.exterior.coords))
    print(row['ID'])
    
    # Definir período de tempo da analise 
    startDate = '2022-01-01'
    endDate = '2023-12-31'

    # Carregar imagens do satélite Sentinel-2
    s2 = ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(poligonos_ee).filterDate(startDate, endDate)
       
    # Função para Calcular a média de cada banda 
    def calculateMean(image):
        mean = image.reduceRegions(
            collection=poligonos_ee,
            reducer=ee.Reducer.mean(),
            scale=10
        )
        return mean

    #media das bandas 
    mean = s2.map(calculateMean).flatten()

    # Converter o resultado em um dataframe
    resultados = mean.getInfo()
    data = []
    for feature in resultados['features']:
        properties = feature['properties']
        properties['cena'] = feature['id']
        properties['ID'] = row['ID']
        data.append(properties)

        dataframe = pd.DataFrame(data)
        df_final = pd.concat([df_final, dataframe])
        

1
2
3
4


## SENTINEL-3

In [12]:
#Armazena as informações das Bandas por Poligono
df_final = pd.DataFrame()

# loop que abre uma feição por vez
for index, row in poligonos.iterrows():
    polygon = row['geometry'] 
    
    # Converter para o tipo lido pelo GEE
    poligonos_ee = ee.Geometry.Polygon(list(polygon.exterior.coords))
    print(row['ID'])
    
    # Definir período de tempo da analise 
    startDate = '2022-01-01'
    endDate = '2023-12-31'

    # Carregar imagens do satélite Sentinel-2
    s2 = ee.ImageCollection('COPERNICUS/S3/OLCI').filterBounds(poligonos_ee).filterDate(startDate, endDate)
       
    # Função para Calcular a média de cada banda 
    def calculateMean(image):
        mean = image.reduceRegions(
            collection=poligonos_ee,
            reducer=ee.Reducer.mean(),
            scale=10
        )
        return mean

    #media das bandas 
    mean = s2.map(calculateMean).flatten()

    # Converter o resultado em um dataframe
    resultados = mean.getInfo()
    data = []
    for feature in resultados['features']:
        properties = feature['properties']
        properties['cena'] = feature['id']
        properties['ID'] = row['ID']
        data.append(properties)

        dataframe = pd.DataFrame(data)
        df_final = pd.concat([df_final, dataframe])

1
2
3
4


In [13]:
dados

,Oa01_radiance,Oa02_radiance,Oa03_radiance,Oa04_radiance,Oa05_radiance,Oa06_radiance,Oa07_radiance,Oa08_radiance,Oa09_radiance,Oa10_radiance,...,Oa16_radiance,Oa17_radiance,Oa18_radiance,Oa19_radiance,Oa20_radiance,Oa21_radiance,quality_flags,cena,ID,data
0,NaN,None,None,None,None,None,None,None,None,NaN,...,None,None,None,None,None,None,NaN,S3A_20220101T125403_20220101T125703_0,1,2022-01-01
2,0.0,None,None,None,None,None,None,None,None,0.0,...,None,None,None,None,None,None,0.0,S3A_20220102T122753_20220102T123053_0,1,2022-01-02
4,NaN,None,None,None,None,None,None,None,None,NaN,...,None,None,None,None,None,None,NaN,S3A_20220105T125019_20220105T125319_0,1,2022-01-05
6,0.0,None,None,None,None,None,None,None,None,0.0,...,None,None,None,None,None,None,0.0,S3A_20220106T122408_20220106T122708_0,1,2022-01-06
8,0.0,None,None,None,None,None,None,None,None,0.0,...,None,None,None,None,None,None,0.0,S3A_20220108T131246_20220108T131546_0,1,2022-01-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1626,NaN,None,None,None,None,None,None,None,None,NaN,...,None,None,None,None,None,None,NaN,S3B_20231218T125228_20231218T125528_0,4,2023-12-18
1630,NaN,None,None,None,None,None,None,None,None,NaN,...,None,None,None,None,None,None,NaN,S3B_20231222T124844_20231222T125144_0,4,2023-12-22
1635,NaN,None,None,None,None,None,None,None,None,NaN,...,None,None,None,None,None,None,NaN,S3B_20231226T124458_20231226T124758_0,4,2023-12-26
1639,0.0,None,None,None,None,None,None,None,None,0.0,...,None,None,None,None,None,None,0.0,S3B_20231229T130724_20231229T131024_0,4,2023-12-29


## ETL no S3:

In [14]:
dados = df_final
dados['data'] = dados['cena'].str[4:12]
dados['data'] = pd.to_datetime(dados['data'], format='%Y%m%d')
dados = dados.drop_duplicates(subset=['ID', 'data'])
dados
dados.to_excel(r"SENTINEL3.xlsx")

## ETL no S2:

In [7]:
dados = df_final[['cena','ID','B1','B11','B12','B2','B3','B4','B5','B6','B7','B8','B8A','B9']]
dados['data'] = dados['cena'].str[:8]
dados['data'] = pd.to_datetime(dados['data'], format='%Y%m%d')
dados = dados.drop_duplicates(subset=['ID', 'data'])
dados = dados.drop('cena', axis =1)
dados

dados.to_excel(r"SENTINEL2.xlsx")


C:\Users\sensix\AppData\Local\Temp\ipykernel_11480\4079407592.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dados['data'] = dados['cena'].str[:8]
C:\Users\sensix\AppData\Local\Temp\ipykernel_11480\4079407592.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dados['data'] = pd.to_datetime(dados['data'], format='%Y%m%d')


# ETL no S1:

In [18]:
dados = df_final
dados['data'] = dados['cena'].str[17:25]
dados['data'] = pd.to_datetime(dados['data'], format='%Y%m%d')
dados = dados.drop_duplicates(subset=['ID', 'data'])
dados
dados.to_excel(r"SENTINEL1.xlsx")
